# GNN Classification From Jaccard Similarity (Step by Step)

This notebook builds a Graph Convolutional Network (GCN) for residue classification (`NAG`, `MAN`, `YZT`, `BGC`) using:
- Node features from key-frequency CSV.
- Graph edges from Jaccard similarity matrix.

We keep each step in a separate cell and explain why we do it.

## Why this pipeline

1. **Features** tell what each protein looks like individually.
2. **Jaccard graph** tells which proteins are similar.
3. **GCN** combines both, often improving ambiguous cases.

This is still a normal supervised classification task: train/val/test + accuracy/F1.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

In [ ]:
# File paths
features_path = r'dnn/localFeatureVect_theta29_dist18_NoFeatureSelection_keyCombine0_header.csv'
jaccard_path = r'jaccard_similarity/theta29_dist18/jaccard_similarity.h5_res_csv/generalised.csv'

assert os.path.exists(features_path), f'Missing: {features_path}'
assert os.path.exists(jaccard_path), f'Missing: {jaccard_path}'

print('Features file:', features_path)
print('Jaccard file:', jaccard_path)

In [ ]:
# Load tabular node features
feature_df = pd.read_csv(features_path)
feature_df = feature_df.rename(columns={feature_df.columns[0]: 'Protein Name'})

print('Feature table shape:', feature_df.shape)
feature_df.head(3)

In [ ]:
# Load custom Jaccard format: name;v1,v2,...,vN
def load_jaccard_generalised(path):
    names = []
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            name, values = line.split(';')
            vec = np.array([float(x) for x in values.split(',')], dtype=np.float32)
            names.append(name)
            rows.append(vec)
    S = np.vstack(rows)
    return names, S

j_names, S = load_jaccard_generalised(jaccard_path)
print('Similarity matrix shape:', S.shape)
print('Min/Max/Mean:', float(S.min()), float(S.max()), float(S.mean()))

In [ ]:
# Basic integrity checks
n = len(j_names)
assert S.shape == (n, n), 'Jaccard matrix must be square'

sym_error = np.abs(S - S.T).max()
print('Max symmetry error:', sym_error)
print('First diagonal values:', np.diag(S)[:5])

feat_names = feature_df['Protein Name'].astype(str).tolist()
assert len(feat_names) == n, 'Feature rows and Jaccard rows must match in count'

# Align feature rows to jaccard row order (safe even if already aligned)
feat_by_name = feature_df.set_index('Protein Name')
missing = [x for x in j_names if x not in feat_by_name.index]
assert not missing, f'Missing feature rows for {len(missing)} nodes'

feature_df = feat_by_name.loc[j_names].reset_index()
print('Rows aligned to Jaccard order.')

In [ ]:
# Create labels from last token after underscore
# Example: 7PNB_o_1_NAG -> NAG
labels_str = [name.split('_')[-1] for name in j_names]
label_counts = pd.Series(labels_str).value_counts()
print(label_counts)

plt.figure(figsize=(5, 3))
label_counts.plot(kind='bar')
plt.title('Class Distribution')
plt.xlabel('Residue label')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Build X and y
X_raw = feature_df.drop(columns=['Protein Name']).values.astype(np.float32)

# Counts are skewed; log1p helps stabilize
X_log = np.log1p(X_raw)
scaler = StandardScaler()
X = scaler.fit_transform(X_log).astype(np.float32)

le = LabelEncoder()
y = le.fit_transform(labels_str).astype(np.int32)

print('X shape:', X.shape)
print('y classes:', dict(zip(le.classes_, range(len(le.classes_)))))

## Train/Val/Test split

We try **grouped split by PDB id** (`name.split('_')[0]`) to reduce leakage.

If there are too few unique groups, we fallback to stratified split.

In [ ]:
indices = np.arange(n)
groups = np.array([name.split('_')[0] for name in j_names])
unique_groups = np.unique(groups)

if len(unique_groups) >= 3:
    gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    train_val_idx, test_idx = next(gss1.split(indices, y, groups=groups))

    # val = 20% of total => 25% of train_val
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    tr_idx_rel, va_idx_rel = next(gss2.split(train_val_idx, y[train_val_idx], groups=groups[train_val_idx]))
    train_idx = train_val_idx[tr_idx_rel]
    val_idx = train_val_idx[va_idx_rel]
    split_mode = 'grouped'
else:
    train_val_idx, test_idx = train_test_split(
        indices, test_size=0.20, random_state=SEED, stratify=y
    )
    train_idx, val_idx = train_test_split(
        train_val_idx, test_size=0.25, random_state=SEED, stratify=y[train_val_idx]
    )
    split_mode = 'stratified-fallback'

print('Split mode:', split_mode)
print('Train/Val/Test:', len(train_idx), len(val_idx), len(test_idx))

## Build graph from Jaccard matrix

The full matrix is dense. We sparsify it with top-`k` neighbors per node.
This avoids over-smoothing and keeps the graph informative.

In [ ]:
k = 10  # try 5, 10, 15
A = np.zeros_like(S, dtype=np.float32)

for i in range(n):
    nbrs = np.argsort(S[i])[-(k + 1):]  # includes self
    for j in nbrs:
        if i != j:
            A[i, j] = S[i, j]

# Make undirected by max weight
A = np.maximum(A, A.T)

# Add self-loops for GCN
np.fill_diagonal(A, 1.0)

num_edges = int((A > 0).sum() - n) // 2
print('Undirected edges (without self-loops):', num_edges)
print('Graph density:', (2 * num_edges) / (n * (n - 1)))

In [ ]:
# Symmetric normalized adjacency: A_hat = D^{-1/2} A D^{-1/2}
deg = A.sum(axis=1)
inv_sqrt_deg = np.power(np.maximum(deg, 1e-8), -0.5)
D_inv_sqrt = np.diag(inv_sqrt_deg)
A_hat = D_inv_sqrt @ A @ D_inv_sqrt

A_hat = A_hat.astype(np.float32)
print('A_hat shape:', A_hat.shape)

## Define a simple 2-layer GCN

Layer rule: `H^{l+1} = activation(A_hat @ H^l @ W^l)`

- Input `H^0 = X`
- Output logits for 4 classes

In [ ]:
class SimpleGCN(tf.keras.Model):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout=0.3):
        super().__init__()
        self.w1 = tf.keras.layers.Dense(hidden_dim, use_bias=False)
        self.w2 = tf.keras.layers.Dense(out_dim, use_bias=False)
        self.dropout = tf.keras.layers.Dropout(dropout)

    def call(self, x, a_hat, training=False):
        h = tf.matmul(a_hat, x)
        h = self.w1(h)
        h = tf.nn.relu(h)
        h = self.dropout(h, training=training)
        h = tf.matmul(a_hat, h)
        out = self.w2(h)
        return out

num_classes = len(le.classes_)
model = SimpleGCN(in_dim=X.shape[1], hidden_dim=64, out_dim=num_classes, dropout=0.35)
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-2, weight_decay=5e-4)

In [ ]:
# Prepare tensors once
X_tf = tf.constant(X, dtype=tf.float32)
A_tf = tf.constant(A_hat, dtype=tf.float32)
y_tf = tf.constant(y, dtype=tf.int32)

train_idx_tf = tf.constant(train_idx, dtype=tf.int32)
val_idx_tf = tf.constant(val_idx, dtype=tf.int32)
test_idx_tf = tf.constant(test_idx, dtype=tf.int32)


def masked_loss_and_metrics(logits, idx):
    y_true = tf.gather(y_tf, idx)
    y_logit = tf.gather(logits, idx)
    loss = tf.reduce_mean(tf.nn.sparse_softmax_cross_entropy_with_logits(labels=y_true, logits=y_logit))

    y_pred = tf.argmax(y_logit, axis=1, output_type=tf.int32).numpy()
    y_true_np = y_true.numpy()
    acc = accuracy_score(y_true_np, y_pred)
    f1 = f1_score(y_true_np, y_pred, average='macro')
    return loss, acc, f1

In [ ]:
# Train with early stopping on validation macro-F1
max_epochs = 400
patience = 50
best_val_f1 = -1.0
best_weights = None
wait = 0

history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}

for epoch in range(1, max_epochs + 1):
    with tf.GradientTape() as tape:
        logits = model(X_tf, A_tf, training=True)
        train_loss, train_acc, train_f1 = masked_loss_and_metrics(logits, train_idx_tf)

    grads = tape.gradient(train_loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

    val_logits = model(X_tf, A_tf, training=False)
    val_loss, val_acc, val_f1 = masked_loss_and_metrics(val_logits, val_idx_tf)

    history['train_loss'].append(float(train_loss.numpy()))
    history['val_loss'].append(float(val_loss.numpy()))
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_weights = model.get_weights()
        wait = 0
    else:
        wait += 1

    if epoch % 20 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | train_loss={train_loss.numpy():.4f} | val_f1={val_f1:.4f}')

    if wait >= patience:
        print(f'Early stopping at epoch {epoch}')
        break

model.set_weights(best_weights)
print('Best validation macro-F1:', round(best_val_f1, 4))

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(history['train_f1'], label='train')
axes[1].plot(history['val_f1'], label='val')
axes[1].set_title('Macro-F1')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final test evaluation
logits = model(X_tf, A_tf, training=False)
test_logits = tf.gather(logits, test_idx_tf)
y_test = tf.gather(y_tf, test_idx_tf).numpy()
y_pred = tf.argmax(test_logits, axis=1, output_type=tf.int32).numpy()

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average='macro')

print('Test accuracy:', round(test_acc, 4))
print('Test macro-F1:', round(test_f1, 4))
print('
Classification report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(np.arange(len(le.classes_)))
ax.set_yticks(np.arange(len(le.classes_)))
ax.set_xticklabels(le.classes_, rotation=45)
ax.set_yticklabels(le.classes_)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha='center', va='center')

fig.colorbar(im)
plt.tight_layout()
plt.show()

## What to try next

- Tune `k` in graph construction (`5, 10, 15`).
- Tune hidden size and dropout.
- Compare with MLP/XGBoost baseline.
- Try using edge threshold instead of top-`k`.
- If grouped split is too strict or impossible, compare with stratified split and report both.